# $\color{#dda}{\text{Parte 2: Tokenización y preprocesamiento}}$

**Temas**
1. Tipos de tokenización de subpalabras: WordPiece, BPE y SentencePiece.
2. `AutoTokenizer`: de texto a tensores y de regreso.
3. Padding, truncation y attention masks.
4. Preprocesar datasets completos con `map()`.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "..")

from hfdemo.config import setup
from hfdemo import plots, tokens
device = setup()

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

pd.set_option("display.max_colwidth", 120)

## $\color{#dda}{\text{1. Por qué subpalabras}}$

| Unidad | Vocabulario | Problema |
|---|---|---|
| Palabra | Enorme (cientos de miles) | Palabras nuevas o raras quedan como `[UNK]` |
| Carácter | Diminuto | Secuencias muy largas; cada token casi no tiene significado |
| **Subpalabra** | 30 k a 250 k | Punto medio: palabras frecuentes enteras, raras partidas en piezas conocidas |

Los tres algoritmos aprenden el vocabulario **a partir de un corpus**; difieren en cómo eligen las piezas:

| Algoritmo | Cómo construye el vocabulario | Marca visible | Modelos |
|---|---|---|---|
| **BPE** (Byte-Pair Encoding) | Empieza con caracteres y fusiona el par **más frecuente**, una y otra vez | `Ġ` = espacio antes (BPE a nivel byte) | GPT-2, RoBERTa, Llama 3 |
| **WordPiece** | Igual que BPE, pero fusiona el par con mayor `freq(ab) / (freq(a)·freq(b))`: premia pares que aparecen juntos más de lo esperado | `##` = continúa la palabra anterior | BERT, DistilBERT |
| **Unigram** (SentencePiece) | Empieza con un vocabulario enorme y **poda** las piezas que menos afectan la verosimilitud del corpus | `▁` = inicio de palabra | T5, XLM-R, mBART |

**SentencePiece** es una librería, no un algoritmo: trata el texto como un flujo crudo (el espacio es un carácter más, `▁`), así que no necesita separar palabras antes. Por eso funciona igual en japonés o chino. Puede usar Unigram o BPE por dentro.

### $\color{#dda}{\text{1.1 Los tres en modelos reales}}$

In [2]:
reales = {
    "WordPiece (bert-base-uncased)": AutoTokenizer.from_pretrained("bert-base-uncased"),
    "BPE (gpt2)": AutoTokenizer.from_pretrained("gpt2"),
    "SentencePiece (xlm-roberta-base)": AutoTokenizer.from_pretrained("xlm-roberta-base"),
}
tokens.tabla_tokens(reales, "Tokenization is unbelievably useful!")

,n,tokens
tokenizer,,
WordPiece (bert-base-uncased),10,token | ##ization | is | un | ##bel | ##ie | ##va | ##bly | useful | !
BPE (gpt2),6,Token | ization | Ġis | Ġunbelievably | Ġuseful | !
SentencePiece (xlm-roberta-base),10,▁To | ken | ization | ▁is | ▁un | beli | eva | bly | ▁useful | !


In [3]:
tokens.tabla_tokens(reales, "La tokenización en español es más cara para modelos entrenados en inglés.")

,n,tokens
tokenizer,,
WordPiece (bert-base-uncased),22,la | token | ##iza | ##cion | en | es | ##pan | ##ol | es | mas | cara | para | model | ##os | en | ##tre | ##nad | ...
BPE (gpt2),26,La | Ġtoken | iz | aci | Ã³n | Ġen | Ġes | pa | Ã± | ol | Ġes | Ġm | Ã¡s | Ġcar | a | Ġpara | Ġmodel | os | Ġent | r...
SentencePiece (xlm-roberta-base),17,▁La | ▁to | ken | ización | ▁en | ▁español | ▁es | ▁más | ▁cara | ▁para | ▁modelos | ▁entre | n | ados | ▁en | ▁ingl...


GPT-2 parte el español en muchas más piezas (y los acentos en bytes sueltos como `Ã`): su vocabulario se aprendió casi solo con inglés. XLM-R, entrenado en 100 idiomas, lo resuelve con menos tokens. Más tokens significa más cómputo y menos texto dentro del límite de contexto.

Fíjate también en `bert-base-uncased`: escribe "tokenizacion" y "espanol". Antes de partir, su **normalizador** pasa todo a minúsculas y quita acentos, así que "más" y "mas" son el mismo token. La normalización es parte del tokenizer, no del modelo.

In [4]:
frases = {
    "inglés": "The cat sat on the mat because it was tired.",
    "español": "El gato se sentó en el tapete porque estaba cansado.",
    "japonés": "猫は疲れていたのでマットの上に座った。",
}
conteos = {n: [len(tk.tokenize(f)) for f in frases.values()] for n, tk in reales.items()}
plots.barras_agrupadas(list(frases), conteos, "Misma frase, distinto costo en tokens",
                       "Idioma de la frase", "Número de tokens")

### $\color{#dda}{\text{1.2 Entrenarlos desde cero}}$
Con la librería `tokenizers` entrenamos los tres algoritmos sobre el mismo corpus (reseñas de cine de *Rotten Tomatoes*) y el mismo tamaño de vocabulario, 2000. Así la única diferencia es el algoritmo.

In [5]:
rt = load_dataset("cornell-movie-review-data/rotten_tomatoes")
corpus = rt["train"]["text"]
print(len(corpus), "reseñas | ejemplo:", corpus[0][:90], "...")

propios = {alg: tokens.entrenar_tokenizer(alg, corpus, vocab_size=2000) for alg in ("BPE", "WordPiece", "Unigram")}

8530 reseñas | ejemplo: the rock is destined to be the 21st century's new " conan " and that he's going to make a  ...


In [6]:
tokens.tabla_tokens(propios, "an unforgettable, overwhelmingly cinematic experience")

,n,tokens
tokenizer,,
BPE,11,"an | un | forgettable | , | over | wh | el | m | ingly | cinematic | experience"
WordPiece,14,"an | unf | ##orge | ##t | ##t | ##able | , | over | ##wh | ##el | ##ming | ##ly | cinematic | experience"
Unigram,9,"▁an | ▁unforgettabl | e | , | ▁overwhelm | ing | ly | ▁cinematic | ▁experience"


Con un vocabulario chico, las palabras raras se parten en piezas frecuentes del corpus. Cada algoritmo corta distinto porque optimiza un criterio distinto.

Las primeras fusiones que aprendió BPE son los pares más frecuentes del inglés:

In [7]:
import json
merges = json.loads(propios["BPE"].to_str())["model"]["merges"]
print([" + ".join(m) for m in merges[:15]])

['t + h', 'i + n', 'a + n', 'e + r', 'th + e', 'r + e', 'e + n', 'o + n', 'a + t', 'i + s', 'e + s', 'i + t', 'in + g', 'o + r', 'o + u']


## $\color{#dda}{\text{2. AutoTokenizer}}$

`AutoTokenizer.from_pretrained(checkpoint)` carga el tokenizer **exacto** con el que se entrenó el modelo. Usar otro tokenizer daría ids que el modelo no entiende.

In [8]:
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
enc = tok("Hugging Face makes NLP easy.")
print(enc.keys())
print("input_ids:     ", enc["input_ids"])
print("tokens:        ", tok.convert_ids_to_tokens(enc["input_ids"]))
print("attention_mask:", enc["attention_mask"])
print("decode:        ", tok.decode(enc["input_ids"]))
print("sin especiales:", tok.decode(enc["input_ids"], skip_special_tokens=True))

KeysView({'input_ids': [101, 17662, 2227, 3084, 17953, 2361, 3733, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]})
input_ids:      [101, 17662, 2227, 3084, 17953, 2361, 3733, 1012, 102]
tokens:         ['[CLS]', 'hugging', 'face', 'makes', 'nl', '##p', 'easy', '.', '[SEP]']
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1]
decode:         [CLS] hugging face makes nlp easy. [SEP]
sin especiales: hugging face makes nlp easy.


El tokenizer agrega solo `[CLS]` al inicio y `[SEP]` al final, que es el formato con el que se preentrenó BERT. Con `return_offsets_mapping` cada token conserva su posición en el texto original, útil para NER o para resaltar palabras:

In [9]:
enc = tok("Hugging Face makes NLP easy.", return_offsets_mapping=True)
[(t, o) for t, o in zip(tok.convert_ids_to_tokens(enc["input_ids"]), enc["offset_mapping"])]

[('[CLS]', (0, 0)),
 ('hugging', (0, 7)),
 ('face', (8, 12)),
 ('makes', (13, 18)),
 ('nl', (19, 21)),
 ('##p', (21, 22)),
 ('easy', (23, 27)),
 ('.', (27, 28)),
 ('[SEP]', (0, 0))]

**Pares de oraciones** (inferencia textual, similitud, QA): se unen con `[SEP]`, y en BERT `token_type_ids` marca a qué oración pertenece cada token.

In [10]:
bert = reales["WordPiece (bert-base-uncased)"]
par = bert("Is it raining?", "Take an umbrella.")
pd.DataFrame({"token": bert.convert_ids_to_tokens(par["input_ids"]), "token_type_id": par["token_type_ids"]}).T

,0,1,2,3,4,5,6,7,8,9,10
token,[CLS],is,it,raining,?,[SEP],take,an,umbrella,.,[SEP]
token_type_id,0,0,0,0,0,0,1,1,1,1,1


## $\color{#dda}{\text{3. Padding, truncation y attention masks}}$

Un tensor necesita que todas las filas midan lo mismo. Los textos no, así que:
- **Padding** rellena las secuencias cortas con `[PAD]`.
- **Truncation** corta las largas a `max_length`.
- **Attention mask** dice al modelo qué posiciones son texto real (1) y cuáles relleno (0).

In [11]:
lote = ["Short one.", "This sentence is quite a bit longer than the first one.", "Medium length sentence here."]

for kwargs in ({"padding": True}, {"padding": "max_length", "max_length": 16}, {"padding": True, "truncation": True, "max_length": 6}):
    out = tok(lote, return_tensors="pt", **kwargs)
    print(kwargs, "-> forma", tuple(out["input_ids"].shape))

{'padding': True} -> forma (3, 14)
{'padding': 'max_length', 'max_length': 16} -> forma (3, 16)
{'padding': True, 'truncation': True, 'max_length': 6} -> forma (3, 6)


In [12]:
out = tok(lote, padding=True, return_tensors="pt")
pd.DataFrame(out["attention_mask"].numpy(), index=lote,
             columns=tok.convert_ids_to_tokens(out["input_ids"][1]))

,[CLS],this,sentence,is,quite,a,bit,longer,than,the,first,one,.,[SEP]
Short one.,1,1,1,1,1,0,0,0,0,0,0,0,0,0
This sentence is quite a bit longer than the first one.,1,1,1,1,1,1,1,1,1,1,1,1,1,1
Medium length sentence here.,1,1,1,1,1,1,1,0,0,0,0,0,0,0


### $\color{#dda}{\text{3.1 Por qué importa la attention mask}}$
Clasificamos "Short one." sola, y luego en un lote junto a una oración larga: con máscara y sin máscara.

In [13]:
from transformers import AutoModelForSequenceClassification
clf = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english").eval()

with torch.no_grad():
    sola = clf(**tok(lote[:1], return_tensors="pt")).logits[0]
    con_mask = clf(**out).logits[0]
    sin_mask = clf(input_ids=out["input_ids"], attention_mask=torch.ones_like(out["input_ids"])).logits[0]

pd.DataFrame([sola.numpy(), con_mask.numpy(), sin_mask.numpy()],
             index=["sola", "en lote, con mask", "en lote, sin mask"], columns=["NEGATIVE", "POSITIVE"]).round(4)

,NEGATIVE,POSITIVE
sola,3.4017,-2.8315
"en lote, con mask",3.4017,-2.8315
"en lote, sin mask",2.0850,-1.8188


Con máscara el resultado es el mismo que la oración sola. Sin máscara el modelo "lee" los `[PAD]` como si fueran texto y los logits cambian. Por eso siempre se pasa `**tokenizer(...)` completo al modelo, no solo `input_ids`.

### $\color{#dda}{\text{3.2 Padding a la izquierda en modelos generativos}}$
Un decodificador (GPT) genera a partir del **último** token, así que el relleno debe ir a la izquierda. GPT-2 además no tiene token de padding: se reutiliza el de fin de texto.

In [14]:
gpt2 = AutoTokenizer.from_pretrained("gpt2", padding_side="left")
gpt2.pad_token = gpt2.eos_token
g = gpt2(["Hi", "Hello there, how are you"], padding=True)
for ids, m in zip(g["input_ids"], g["attention_mask"]):
    print(gpt2.convert_ids_to_tokens(ids), m)

['<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', '<|endoftext|>', 'Hi'] [0, 0, 0, 0, 0, 1]
['Hello', 'Ġthere', ',', 'Ġhow', 'Ġare', 'Ġyou'] [1, 1, 1, 1, 1, 1]


### $\color{#dda}{\text{3.3 Textos más largos que el modelo}}$
Truncar pierde información. Para documentos largos se parte en ventanas con traslape (`stride`), como se hace en QA:

In [15]:
largo = "Transformers process text in fixed windows, so long documents must be split into overlapping chunks before inference."
ventanas = tok(largo, max_length=14, truncation=True, stride=4, return_overflowing_tokens=True)
for ids in ventanas["input_ids"]:
    print(tok.decode(ids))

[CLS] transformers process text in fixed windows, so long documents must be [SEP]
[CLS] long documents must be split into overlapping chunks before inference. [SEP]


## $\color{#dda}{\text{4. Preprocesar datasets con map()}}$

`Dataset.map(función)` aplica una función a cada ejemplo y guarda el resultado en Arrow (en disco y con caché). Con `batched=True` la función recibe lotes de 1000 ejemplos, y el tokenizer en Rust los procesa en paralelo.

In [16]:
rt

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [17]:
def tokenizar(lote):
    return tok(lote["text"], truncation=True, max_length=128)

import time
for batched in (False, True):
    t0 = time.perf_counter()
    rt_tok = rt["train"].map(tokenizar, batched=batched, load_from_cache_file=False)
    print(f"batched={batched!s:5s}: {time.perf_counter() - t0:.2f} s")

batched=False: 2.06 s


batched=True : 0.56 s


In [18]:
rt_tok = rt.map(tokenizar, batched=True, remove_columns=["text"])
rt_tok

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
})

In [19]:
print(rt_tok["train"][0])

{'label': 1, 'input_ids': [101, 1996, 2600, 2003, 16036, 2000, 2022, 1996, 7398, 2301, 1005, 1055, 2047, 1000, 16608, 1000, 1998, 2008, 2002, 1005, 1055, 2183, 2000, 2191, 1037, 17624, 2130, 3618, 2084, 7779, 29058, 8625, 13327, 1010, 3744, 1011, 18856, 19513, 3158, 5477, 4168, 2030, 7112, 16562, 2140, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


`map` agregó `input_ids` y `attention_mask` a los tres splits y quitamos `text`, que el modelo no usa. Otras opciones útiles: `num_proc` para varios procesos, `filter()` para quitar ejemplos y `with_format("torch")` para obtener tensores.

### $\color{#dda}{\text{4.1 Padding dinámico}}$
No rellenamos en `map`: cada ejemplo conserva su longitud real. El relleno se hace **por lote** al entrenar con `DataCollatorWithPadding`, hasta la longitud del ejemplo más largo **del lote**, no del dataset.

In [20]:
from transformers import DataCollatorWithPadding
collator = DataCollatorWithPadding(tokenizer=tok)

muestras = [rt_tok["train"][i] for i in range(4)]
batch = collator(muestras)
print({k: tuple(v.shape) for k, v in batch.items()})

{'input_ids': (4, 52), 'token_type_ids': (4, 52), 'attention_mask': (4, 52), 'labels': (4,)}


In [21]:
longitudes = rt_tok["train"]["input_ids"]
lotes = [longitudes[i:i + 16] for i in range(0, len(longitudes), 16)]
estatico = 128 * len(longitudes)
dinamico = sum(16 * max(len(x) for x in l) for l in lotes)
real = sum(len(x) for x in longitudes)
plots.barras_agrupadas(["padding fijo a 128", "padding dinámico (lotes de 16)", "tokens reales"],
                       {"posiciones": [estatico, dinamico, real]},
                       "Posiciones que procesa el modelo en un epoch", "Estrategia", "Número de posiciones")

In [22]:
print(f"El padding dinámico procesa {estatico / dinamico:.1f}x menos posiciones que el padding fijo.")

El padding dinámico procesa 2.6x menos posiciones que el padding fijo.


## $\color{#dda}{\text{Resumen}}$

- BPE fusiona por frecuencia, WordPiece por "pegajosidad" del par, Unigram poda desde un vocabulario grande. SentencePiece trata el espacio como carácter y no depende del idioma.
- El tokenizer va **siempre** emparejado con su modelo: `AutoTokenizer.from_pretrained(mismo_checkpoint)`.
- Padding + attention mask permiten lotes; sin máscara, el relleno contamina la salida.
- `map(batched=True)` tokeniza datasets completos rápido y con caché; el padding dinámico con `DataCollatorWithPadding` ahorra cómputo.